# FaceNet from scratch: face verification

FaceNet NN2 implemented in PyTorch, trained with triplet loss and evaluated on Labeled Faces in the Wild.

In [1]:
import copy
import os
import random
from typing import Literal, Callable

from PIL import Image
import numpy as np
from sklearn.metrics import roc_curve, auc
from pytorch_metric_learning import losses, miners, samplers
import torch
from torch.amp import autocast, GradScaler
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import LinearLR
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import ImageFolder
from torchvision import transforms
import torchvision.models as models

## 1. Building architecture (FaceNet NN2)

In [2]:
class BasicConv2d(nn.Module):
    def __init__(
            self,
            in_channels: int,
            out_channels: int,
            kernel_size: int,
            **kwargs,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, 
            out_channels, 
            kernel_size,
            bias=False, 
            **kwargs,
        )
        self.batch_norm = nn.BatchNorm2d(out_channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv(x)
        x = self.batch_norm(x)
        return F.relu(x, inplace=True)


class L2Pool(nn.Module):
    def __init__(self, kernel_size: int, stride: int, padding: int):
        super().__init__()
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        sum_squared = F.avg_pool2d(
            x.pow(2), 
            self.kernel_size, 
            self.stride, 
            self.padding, 
            divisor_override=1,
        )
        return torch.sqrt(sum_squared + 1e-6)


class NN2Inception(nn.Module):
    def __init__(
        self,
        in_channels: int,
        ch1x1: int,
        ch3x3_red: int,
        ch3x3: int,
        ch5x5_red: int,
        ch5x5: int,
        pool_proj: int,
        pool_type: Literal["max_pool", "l2_pool"] = "max_pool"
    ):
        super().__init__()
        self.branch1x1 = BasicConv2d(in_channels, ch1x1, kernel_size=1)
        self.branch3x3 = nn.Sequential(
            BasicConv2d(in_channels, ch3x3_red, kernel_size=1),
            BasicConv2d(ch3x3_red, ch3x3, kernel_size=3, padding=1),
        )
        self.branch5x5 = nn.Sequential(
            BasicConv2d(in_channels, ch5x5_red, kernel_size=1),
            BasicConv2d(ch5x5_red, ch5x5, kernel_size=3, padding=1),
        )

        if pool_type == "max_pool":
            pool_layer = nn.MaxPool2d(kernel_size=3, stride=1, padding=1)
        if pool_type == "l2_pool":
            pool_layer = L2Pool(kernel_size=3, stride=1, padding=1)
        
        self.branch_pool = nn.Sequential(
            pool_layer,
            BasicConv2d(in_channels, pool_proj, kernel_size=1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.cat([
            self.branch1x1(x),
            self.branch3x3(x),
            self.branch5x5(x),
            self.branch_pool(x),
        ], dim=1)


class InceptionReduction(nn.Module):
    def __init__(
        self, 
        in_channels: int, 
        ch3x3_red: int, 
        ch3x3: int, 
        ch5x5_red: int, 
        ch5x5: int,
    ):
        super().__init__()
        self.branch3x3 = nn.Sequential(
            BasicConv2d(in_channels, ch3x3_red, kernel_size=1),
            BasicConv2d(ch3x3_red, ch3x3, kernel_size=3, stride=2, padding=1)
        )
        
        self.branch5x5 = nn.Sequential(
            BasicConv2d(in_channels, ch5x5_red, kernel_size=1),
            BasicConv2d(ch5x5_red, ch5x5, kernel_size=3, stride=2, padding=1)
        )
        
        self.branch_pool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.cat([
            self.branch3x3(x), 
            self.branch5x5(x), 
            self.branch_pool(x),
        ], dim=1)


class FaceNetNN2(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.conv1 = BasicConv2d(3, 64, kernel_size=7, stride=2, padding=3)
        self.pool1 = nn.MaxPool2d(3, stride=2, ceil_mode=True)
        
        self.conv2 = nn.Sequential(
            BasicConv2d(64, 64, kernel_size=1),
            BasicConv2d(64, 192, kernel_size=3, padding=1),
        )
        self.pool2 = nn.MaxPool2d(3, stride=2, ceil_mode=True)
        
        self.inception3a = NN2Inception(192, 64, 96, 128, 16, 32, 32, "max_pool")
        self.inception3b = NN2Inception(256, 64, 96, 128, 32, 64, 64, "l2_pool")
        self.inception3c = InceptionReduction(320, 128, 256, 32, 64)
        
        self.inception4a = NN2Inception(640, 256, 96, 192, 32, 64, 128, "l2_pool")
        self.inception4b = NN2Inception(640, 224, 112, 224, 32, 64, 128, "l2_pool")
        self.inception4c = NN2Inception(640, 192, 128, 256, 32, 64, 128, "l2_pool")
        self.inception4d = NN2Inception(640, 160, 144, 288, 32, 64, 128, "l2_pool")
        self.inception4e = InceptionReduction(640, 160, 256, 64, 128)
        
        self.inception5a = NN2Inception(1024, 384, 192, 384, 48, 128, 128, "l2_pool")
        self.inception5b = NN2Inception(1024, 384, 192, 384, 48, 128, 128, "max_pool")
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.full_layer = nn.Linear(1024, 128)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pool1(self.conv1(x))
        x = self.pool2(self.conv2(x))
        
        x = self.inception3a(x)
        x = self.inception3b(x)
        x = self.inception3c(x)
        
        x = self.inception4a(x)
        x = self.inception4b(x)
        x = self.inception4c(x)
        x = self.inception4d(x)
        x = self.inception4e(x)
        
        x = self.inception5a(x)
        x = self.inception5b(x)
        
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.full_layer(x)
        
        return F.normalize(x, p=2, dim=1)

## 2. Preparing evaluation

Labelled Faces dataset is designed for the problem of face verification -- given 2 images predict if they capture the same person or 2 different people. To teach model answer this question I need a dataset of images pairs:

In [3]:
class PairsDataset(Dataset):
    def __init__(
        self,
        root_dir: str,
        transform: Callable[[Image.Image], torch.Tensor],
        pairs_num=1000,
        random_state: int | None = None,
    ):
        self.root_dir = root_dir
        self.pairs_num = pairs_num
        self.transform = transform
        self.pairs = []
        self.pair_labels = []
        self.rng = random.Random(random_state)

        self._load_paths()
        self._generate_pairs(is_positive=1)
        self._generate_pairs(is_positive=0)

    def _load_paths(self) -> None:
        self.identities = os.listdir(self.root_dir)
        self.image_paths: dict[str, list[str]] = \
            {entity: [] for entity in self.identities}

        for entity in self.identities:
            entity_dir = os.path.join(self.root_dir, entity)
            for img_name in os.listdir(entity_dir):
                self.image_paths[entity].append(os.path.join(entity_dir, img_name))
                
        self.positive_pair_candidates = [
            entity for entity in self.identities 
            if len(self.image_paths[entity]) >= 2
        ]
    
    def _sample_pair(self, is_positive: bool) -> tuple[str, str]:
        if is_positive:
            entity = self.rng.choice(self.positive_pair_candidates)
            img1, img2 = self.rng.sample(self.image_paths[entity], 2)
        else:
            entity1, entity2 = self.rng.sample(self.identities, 2)
            img1 = self.rng.choice(self.image_paths[entity1])
            img2 = self.rng.choice(self.image_paths[entity2])
        
        return tuple(sorted((img1, img2)))

    def _generate_pairs(self, is_positive: int) -> None:
        seen_pairs: set[tuple[str, str]] = set()
        target_count = self.pairs_num // 2
        current_count = 0
        
        while current_count < target_count:
            pair = self._sample_pair(is_positive)
            if pair not in seen_pairs:
                seen_pairs.add(pair)
                self.pairs.append((pair[0], pair[1]))
                self.pair_labels.append(is_positive)
                current_count += 1

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        path1, path2 = self.pairs[idx]
        label = self.pair_labels[idx]

        img1 = self.transform(Image.open(path1).convert('RGB'))
        img2 = self.transform(Image.open(path2).convert('RGB'))
            
        return img1, img2, torch.tensor(label, dtype=torch.long)

FaceNet model forward method returns embeddings for input image. I want to predict that if 2 images have embeddings close to each other (up to some threshold) then they refer to the same person otherwise to different people. I am going to use ROC-AUC as evaluation metrics to be able to choose the best distance threshold (and as I generated dataset of pairs explicitly so there are half positive and half negative pairs, ROC-AUC is a good metric here).

In [4]:
def calculate_distances(
        model: nn.Module,
        val_loader: DataLoader,
        device: str,
) -> tuple[np.ndarray, np.ndarray]:
    distances: list[np.ndarray] = []
    labels: list[np.ndarray] = []

    for image1_batch, image2_batch, labels_batch in val_loader:
        image1_batch = image1_batch.to(device)
        image2_batch = image2_batch.to(device)
        
        embed1 = model(image1_batch)
        embed2 = model(image2_batch)

        batch_distances = (embed1 - embed2).norm(p=2, dim=1)

        distances.append(batch_distances.cpu().numpy())
        labels.append(labels_batch.numpy())
        
    distances_np = np.concatenate(distances)
    labels_np = np.concatenate(labels)
    return distances_np, labels_np


@torch.no_grad()
def evaluate_verification(
    model: nn.Module, 
    val_loader: DataLoader, 
    device: Literal["cpu", "cuda"],
) -> tuple[float, float]:
    model.eval()
    
    distances, labels = calculate_distances(model, val_loader, device)

    false_pos_rate, true_pos_rate, thresholds = roc_curve(labels, -distances)
    roc_auc = auc(false_pos_rate, true_pos_rate)
    
    best_threshold_idx = np.argmax(true_pos_rate - false_pos_rate)
    best_threshold = -thresholds[best_threshold_idx]

    labels_pred = (distances < best_threshold).astype(int)
    accuracy = np.mean(labels_pred == labels)
    
    return roc_auc, accuracy

## 3. Building training loop

In [5]:
def train_epoch(
    model: nn.Module, 
    dataloader: DataLoader,
    optimizer: optim.Optimizer, 
    miner: miners.BaseMiner, 
    loss_func: losses.BaseMetricLossFunction, 
    device: Literal["cpu", "cuda"], 
    scaler: GradScaler,
    verbose: bool,
) -> float:
    model.train()
    running_loss = 0.0
    total_samples = 0
    
    for batch_idx, (images, labels) in enumerate(dataloader):
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()

        with autocast(device_type=device, dtype=torch.float16):
            embeddings = model(images)
        
        embeddings = embeddings.float()
        hard_pairs = miner(embeddings, labels)
        loss = loss_func(embeddings, labels, hard_pairs)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        total_samples += batch_size
        
        if verbose and batch_idx % 10 == 0:
            print(f"Batch {batch_idx}, Training loss: {loss.item():.4f}")
    
    epoch_loss = running_loss / total_samples
    return epoch_loss

In [6]:
def load_train(
        input_size: tuple[int, int],
        train_data_path: str,
        samples_per_entity: int,
        batch_size: int,
        num_workers: int,
) -> DataLoader:
    train_transform = transforms.Compose([
        transforms.Resize(input_size),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])
    train_dataset = ImageFolder(
        root=train_data_path,
        transform=train_transform,
    )
    sampler = samplers.MPerClassSampler(
        labels=train_dataset.targets, 
        m=samples_per_entity, 
        batch_size=batch_size, 
        length_before_new_iter=len(train_dataset),
    )
    return DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        sampler=sampler, 
        num_workers=num_workers,
        pin_memory=True
    )

def load_validation(
        input_size: tuple[int, int],
        val_data_path: str,
        eval_pairs_num: int,
        batch_size: int,
        num_workers: int,
) -> DataLoader:
    val_transform = transforms.Compose([
        transforms.Resize(input_size),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.5, 0.5, 0.5],
            std=[0.5, 0.5, 0.5],
        )
    ])
    val_dataset = PairsDataset(
        root_dir=val_data_path,
        transform=val_transform,
        pairs_num=eval_pairs_num,
    )
    
    return DataLoader(
        val_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=num_workers,
        pin_memory=True
    )


def load_data(
        train_data_path: str,
        val_data_path: str,
        input_size: tuple[int, int],
        samples_per_entity: int,
        batch_size: int,
        num_workers: int,
        eval_pairs_num: int,
):
    train_loader = load_train(
        input_size,
        train_data_path,
        samples_per_entity,
        batch_size, 
        num_workers, 
    )
    val_loader = load_validation(
        input_size,
        val_data_path,
        eval_pairs_num,
        batch_size,
        num_workers,
    )
    return train_loader, val_loader


def train_face_recognition(
        model: nn.Module,
        train_data_path: str,
        val_data_path: str,
        input_size=(224, 224),
        batch_size=64,
        samples_per_entity=4,
        num_workers=4,
        eval_pairs_num=1_000,
        lr=1e-3,
        margin=0.2,
        num_epochs=10,
        warmup_epochs=3,
        verbose=True,
        adam_decay=1e-8,
) -> None:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Training on device ", device)

    train_loader, val_loader = load_data(
        train_data_path,
        val_data_path,
        input_size,
        samples_per_entity,
        batch_size,
        num_workers,
        eval_pairs_num,
    )

    model = model.to(device)
    optimizer = optim.AdamW(
        params=model.parameters(),
        lr=lr,
        weight_decay=adam_decay,
    )
    scheduler = LinearLR(
        optimizer, 
        start_factor=0.01,
        total_iters=warmup_epochs
    )
    scaler = GradScaler(device=device)
    
    miner = miners.TripletMarginMiner(margin=margin, type_of_triplets="semihard")
    loss_func = losses.TripletMarginLoss(margin=margin)
    
    best_roc_auc = 0.0
    best_weights = None
    for epoch in range(num_epochs):
        epoch_loss = train_epoch(
            model,
            train_loader,
            optimizer,
            miner, 
            loss_func,
            device,
            scaler,
            verbose,
        )

        roc_auc, val_accuracy = evaluate_verification(
            model,
            val_loader,
            device,
        )
        
        if (roc_auc > best_roc_auc):
            best_roc_auc = roc_auc
            best_weights = copy.deepcopy(model.state_dict())

        print(
            f"Epoch {epoch+1}/{num_epochs}, training loss {epoch_loss:.4f}. "
            f"Validation ROC-AUC {roc_auc:.4f}, best threshold accuracy {val_accuracy:.4f}."
        )

        scheduler.step()
    model.load_state_dict(best_weights)
    print("Best validation weights restored.")


## 4. Architecture from the box to compare with

In [7]:
class PretrainedFaceResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, 128)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.backbone(x)
        return F.normalize(x, p=2, dim=1)

## 5. Evaluating model

For training we need a dataset that has enough images per one person, that is easy to download and that doesn't take too much space.

In Labelled Faces in the Wild most of people have only one image, so this dataset doesn't fit for training.

I used dataset with 17,000 images of 105 celebrities. It is not enough for groundbreaking scores, but let's hope we will see a significant improvement in learning.

In [8]:
input_size = (160, 160)
custom_model = FaceNetNN2()
train_face_recognition(
    model=custom_model,
    train_data_path="train_faces_dataset",
    val_data_path="val_faces_dataset",
    input_size=input_size,
    num_epochs=20,
    verbose=False,
    batch_size=128,
    lr=1e-3,
    warmup_epochs=3,
)

Training on device  cuda
Epoch 1/20, training loss 0.1059. Validation ROC-AUC 0.5435, best threshold accuracy 0.5560.
Epoch 2/20, training loss 0.1033. Validation ROC-AUC 0.5921, best threshold accuracy 0.5710.
Epoch 3/20, training loss 0.1007. Validation ROC-AUC 0.6511, best threshold accuracy 0.6220.
Epoch 4/20, training loss 0.1002. Validation ROC-AUC 0.6850, best threshold accuracy 0.6390.
Epoch 5/20, training loss 0.0985. Validation ROC-AUC 0.7635, best threshold accuracy 0.6940.
Epoch 6/20, training loss 0.0980. Validation ROC-AUC 0.7778, best threshold accuracy 0.7180.
Epoch 7/20, training loss 0.0973. Validation ROC-AUC 0.7767, best threshold accuracy 0.7170.
Epoch 8/20, training loss 0.0976. Validation ROC-AUC 0.7448, best threshold accuracy 0.6930.
Epoch 9/20, training loss 0.0968. Validation ROC-AUC 0.8070, best threshold accuracy 0.7400.
Epoch 10/20, training loss 0.0965. Validation ROC-AUC 0.7842, best threshold accuracy 0.7220.
Epoch 11/20, training loss 0.0962. Validatio

Validation ROC-AUC grows up to ~0.8 but then jumps back and forth around this mark with accuracy jumping around 0.75.

Increasing regularization on weights in Adam optimizer didn't change that.

In [9]:
proven_model = PretrainedFaceResNet()
train_face_recognition(
    model=proven_model,
    train_data_path="train_faces_dataset",
    val_data_path="val_faces_dataset",
    input_size=input_size,
    num_epochs=20,
    verbose=False,
    batch_size=128,
    lr=1e-5,
    adam_decay=1e-6,
)

Training on device  cuda
Epoch 1/20, training loss 0.1159. Validation ROC-AUC 0.6054, best threshold accuracy 0.5740.
Epoch 2/20, training loss 0.1075. Validation ROC-AUC 0.7177, best threshold accuracy 0.6580.
Epoch 3/20, training loss 0.1032. Validation ROC-AUC 0.7660, best threshold accuracy 0.7090.
Epoch 4/20, training loss 0.0999. Validation ROC-AUC 0.7810, best threshold accuracy 0.7140.
Epoch 5/20, training loss 0.0982. Validation ROC-AUC 0.8013, best threshold accuracy 0.7310.
Epoch 6/20, training loss 0.0973. Validation ROC-AUC 0.8223, best threshold accuracy 0.7490.
Epoch 7/20, training loss 0.0964. Validation ROC-AUC 0.8297, best threshold accuracy 0.7520.
Epoch 8/20, training loss 0.0960. Validation ROC-AUC 0.8394, best threshold accuracy 0.7650.
Epoch 9/20, training loss 0.0952. Validation ROC-AUC 0.8414, best threshold accuracy 0.7620.
Epoch 10/20, training loss 0.0942. Validation ROC-AUC 0.8469, best threshold accuracy 0.7630.
Epoch 11/20, training loss 0.0940. Validatio

Validation ROC-AUC of pretrained ResNet model reaches 0.857

## 6. Testing on Labelled Faces in the Wild.

In [12]:
lfw_loader = load_validation(
    input_size=input_size,
    val_data_path="lfw-deepfunneled",
    eval_pairs_num=6_000,
    batch_size=256,
    num_workers=4,
)

In [14]:
roc_auc, accuracy = evaluate_verification(
    model=custom_model,
    val_loader=lfw_loader,
    device="cuda",
)
print(
    f"Custom FaceNet NN2.\n Labelled Faces roc_auc: {roc_auc:.4f}, "
    f"best threshold accuracy: {accuracy:.4f}."
)

Custom FaceNet NN2.
 Labelled Faces roc_auc: 0.7239, best threshold accuracy: 0.6720.


In [15]:
roc_auc, accuracy = evaluate_verification(
    model=proven_model,
    val_loader=lfw_loader,
    device="cuda",
)
print(
    f"ResNet retrained for faces recognition.\n"
    f"Labelled Faces roc_auc: {roc_auc:.4f}, "
    f"best threshold accuracy: {accuracy:.4f}"
)

ResNet retrained for faces recognition.
Labelled Faces roc_auc: 0.8350, best threshold accuracy: 0.7602
